# 03 - Antrenarea și compararea modelelor

Acest notebook compară trei algoritmi de clasificare text:

1. Multinomial Naive Bayes
2. Logistic Regression
3. LinearSVC

Scopul este să se observe performanța fiecărui model și să se justifice alegerea modelului final.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import joblib

CLEAN_PATH = Path('products_cleaned.csv')
if not CLEAN_PATH.exists():
    CLEAN_PATH = Path('/mnt/data/products_cleaned.csv')

if CLEAN_PATH.exists():
    data = pd.read_csv(CLEAN_PATH)
else:
    raw_path = Path('products.csv')
    if not raw_path.exists():
        raw_path = Path('/mnt/data/products.csv')
    data = pd.read_csv(raw_path)
    data.columns = data.columns.str.strip()
    data = data[['Product Title', 'Category Label']].dropna().copy()
    data = data.rename(columns={'Product Title': 'product_title', 'Category Label': 'category'})
    data['clean_title'] = data['product_title'].astype(str).str.lower().str.replace(r'[^a-z0-9\s]', ' ', regex=True).str.replace(r'\s+', ' ', regex=True).str.strip()
    data['category'] = data['category'].astype(str).str.strip()
    data = data[data['clean_title'].str.len() > 0].copy()

data.head()

## 1. Împărțirea datelor

In [ ]:
X = data['clean_title']
y = data['category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Train:', X_train.shape)
print('Test:', X_test.shape)

## 2. Definirea modelelor

Toate modelele folosesc aceeași reprezentare a textului: `TfidfVectorizer`.

Am folosit aceleași date de train/test pentru toate modelele, astfel încât comparația să fie corectă.

In [ ]:
models = {
    'Multinomial Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000, solver='lbfgs'),
    'LinearSVC': LinearSVC(random_state=42, max_iter=3000)
}

results = []
trained_pipelines = {}

for model_name, model in models.items():
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            max_features=30000
        )),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    trained_pipelines[model_name] = pipeline

    results.append({
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision weighted': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'Recall weighted': recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'F1 weighted': f1_score(y_test, y_pred, average='weighted', zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values(by='F1 weighted', ascending=False)
results_df

## 3. Compararea rezultatelor

In [ ]:
ax = results_df.set_index('Model')[['Accuracy', 'F1 weighted']].plot(kind='bar', figsize=(9, 5))
plt.title('Compararea modelelor')
plt.ylabel('Scor')
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Rapoarte de clasificare pentru fiecare model

In [ ]:
for model_name, pipeline in trained_pipelines.items():
    print('=' * 80)
    print(model_name)
    print('=' * 80)
    y_pred = pipeline.predict(X_test)
    print(classification_report(y_test, y_pred, zero_division=0))

## 5. Confusion Matrix Comparison

Matricea de confuzie arată unde modelul clasifică corect și unde confundă unele categorii între ele.

In [ ]:
for model_name, pipeline in trained_pipelines.items():
    y_pred = pipeline.predict(X_test)
    labels = sorted(y_test.unique())
    cm = confusion_matrix(y_test, y_pred, labels=labels)

    fig, ax = plt.subplots(figsize=(10, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, xticks_rotation=45, values_format='d')
    plt.title(f'Matrice de confuzie - {model_name}')
    plt.tight_layout()
    plt.show()

## 6. Alegerea modelului final

Modelul final este ales pe baza scorului F1 weighted și a acurateței.

`LinearSVC` este, de obicei, foarte eficient pentru clasificarea textelor scurte, deoarece funcționează bine cu reprezentări sparse de tip TF-IDF. În acest proiect, alegerea finală trebuie justificată prin tabelul de rezultate generat mai sus.

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_pipeline = trained_pipelines[best_model_name]

print('Cel mai bun model după F1 weighted este:', best_model_name)
print(results_df.iloc[0])

## 7. Salvarea modelului final

In [ ]:
MODEL_PATH = Path('best_product_classifier.joblib')
joblib.dump(best_pipeline, MODEL_PATH)
print(f'Model salvat: {MODEL_PATH.resolve()}')

## 8. Testare rapidă cu produse noi

In [ ]:
examples = [
    'samsung galaxy s23 ultra 256gb black',
    'bosch washing machine 8kg 1400 rpm',
    'intel core i7 processor',
    'lg 55 inch smart tv 4k'
]

predictions = best_pipeline.predict(examples)

for text, pred in zip(examples, predictions):
    print(f'{text} -> {pred}')

## Concluzii finale

- Au fost comparate trei modele de clasificare: Multinomial Naive Bayes, Logistic Regression și LinearSVC.
- Modelele au fost evaluate cu Accuracy, Precision, Recall și F1 weighted.
- Secțiunea Confusion Matrix Comparison permite observarea categoriilor unde apar cele mai multe confuzii.
- Modelul final se salvează în fișierul `best_product_classifier.joblib` și poate fi reutilizat într-un script `.py`.
- Dacă `LinearSVC` are cel mai bun scor sau un scor foarte apropiat de cel mai bun, alegerea lui este justificată prin performanță și eficiență pentru clasificarea textului.